### ==============================================================================
## Processing of Moving Vessel Profiler Data - code 4_smoothing_and_LOESS_filter
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es) & Bàrbara Barceló-Llull (bbarcelo@imedea.uib-csic.es)
### Data from FaSt-SWOT experiment
# 
**DESCRIPTION**:
 This script applies a differentiated smoothing technique (Split Smoothing) 
 to the lag-corrected data generated in Step 3. 
 - Temperature: Undergoes a gentle 5-point rolling median filter to remove 
   residual electrical spikes while preserving fine-scale physical structures.
 - Salinity: Undergoes both the median filter and a physical LOESS 
   (Locally Estimated Scatterplot Smoothing) filter adjusted to a 10m 
   vertical scale. This effectively dampens the high-frequency noise inherent 
   to thermal mass and conductivity cell mismatches.
#
 INPUT: Lag-corrected NetCDF files (*_step3_lag.nc).
#
 OUTPUT: Smoothed NetCDF files (*_step4_loess.nc).
### ==============================================================================

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import statsmodels.api as sm
import gsw
import warnings

warnings.filterwarnings("ignore")

# =========================
# 1. CONFIGURATION
# =========================
BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing")

# Input directories synchronized with Step 3 outputs
INPUT_DIRS = {
    "Leg1": BASE_ROOT / "Data" / "Leg1" / "processed_step3_lag",
    "Leg2": BASE_ROOT / "Data" / "Leg2" / "processed_step3_lag",
}

OUT_SUBDIR = "processed_step4_loess"

# --- PARAMETERS ---
MEDIAN_WINDOW = 5              # Scans (removes electrical spikes)
VERTICAL_SCALE_SALT = 10.0     # meters (smoothing scale for Salinity)

# =========================
# 2. FUNCTIONS
# =========================

def apply_loess(data, depth, scale_meters):
    """
    Apply LOESS adjusting the parameter 'frac' according to the profile depth.
    """
    mask = np.isfinite(data) & np.isfinite(depth)
    if mask.sum() < 20: return data 
    
    y = data[mask]
    x = depth[mask]
    
    # Compute vertical range and adjust 'frac' to ensure a consistent physical smoothing scale
    z_range = x.max() - x.min()
    if z_range < scale_meters: 
        frac = 0.8 
    else:
        frac = scale_meters / z_range
        frac = np.clip(frac, 0.02, 0.5) 
        
    # Apply LOWESS (it=0 for speed, return_sorted=False to maintain order)
    smoothed = sm.nonparametric.lowess(y, x, frac=frac, it=0, return_sorted=False)
    
    out = np.full_like(data, np.nan)
    out[mask] = smoothed
    
    return out

# =========================
# 3. MAIN LOOP
# =========================
print(f"--- STEP 4: SMOOTHING (T=Median, S=LOESS {VERTICAL_SCALE_SALT}m) ---")

for leg_name, input_dir in INPUT_DIRS.items():
    if not input_dir.exists():
        print(f"⚠️ Skipping {leg_name}: Path does not exist {input_dir}")
        continue

    # Create output directory
    out_dir = input_dir.parent / OUT_SUBDIR
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # Search for files generated in Step 3
    files = sorted(input_dir.glob("*_step3_lag.nc"))
    print(f"\n[{leg_name.upper()}] Processing {len(files)} files...")
    
    count = 0
    for f in files:
        try:
            ds = xr.open_dataset(f)
            
            # 1. LOAD VARIABLES
            pres = ds['pressure'].values
            
            # Temperature: Priority to the lag-corrected variable (t1_lagged)
            if 't1_lagged' in ds:
                temp = ds['t1_lagged'].values
            else:
                temp = ds['t1'].values
                
            # Salinity: Priority to the lag-corrected variable (salinity_lagged)
            if 'salinity_lagged' in ds:
                salt = ds['salinity_lagged'].values
            else:
                # Fallback calculating with gsw if the variable does not exist
                salt = gsw.SP_from_C(ds['c1'].values, temp, pres)
            
            # 2. SMOOTHING TEMPERATURE (Median only for despiking)
            t_series = pd.Series(temp)
            t_smooth = t_series.rolling(window=MEDIAN_WINDOW, center=True, min_periods=1).median().values
            
            # 3. SMOOTHING SALINITY (Median + LOESS)
            # A) Previous median to remove conductivity spikes
            s_series = pd.Series(salt)
            s_median = s_series.rolling(window=MEDIAN_WINDOW, center=True, min_periods=1).median().values
            
            # B) LOESS
            s_smooth = apply_loess(s_median, pres, scale_meters=VERTICAL_SCALE_SALT)
            
            # 4. SAVE RESULTS
            ds_out = ds.copy(deep=True)
            
            ds_out['t1_smooth'] = (('scan',), t_smooth)
            ds_out['salinity_smooth'] = (('scan',), s_smooth)
            
            # Metadata for the new variables
            ds_out['t1_smooth'].attrs = {
                'units': 'degC', 
                'comment': f'Despiked (Median window {MEDIAN_WINDOW}) post-lag correction.'
            }
            ds_out['salinity_smooth'].attrs = {
                'units': 'PSU', 
                'comment': f'Despiked + LOESS ({VERTICAL_SCALE_SALT}m vertical scale) post-lag correction.'
            }
            
            ds_out.attrs['step4_status'] = 'Smoothed_LOESS'
            
            # Save with the new name
            out_name = f.name.replace('_step3_lag.nc', '_step4_loess.nc')
            ds_out.to_netcdf(out_dir / out_name)
            
            count += 1
            ds.close()
            
        except Exception as e:
            print(f"❌ Error in {f.name}: {e}")
            
    print(f"✅ Generated {count} files in: {out_dir}")

print("\n--- STEP 4 COMPLETED ---")

c:\Users\ASUS\anaconda3\envs\env_elisabet\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\ASUS\anaconda3\envs\env_elisabet\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


--- STEP 4: SMOOTHING (T=Median, S=LOESS 10.0m) ---

[LEG1] Processing 596 files...
✅ Generated 596 files in: C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing\Data\Leg1\processed_step4_loess

[LEG2] Processing 307 files...
✅ Generated 307 files in: C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing\Data\Leg2\processed_step4_loess

--- STEP 4 COMPLETED ---
